# Module 2 — Agentic RAG: A LangGraph Retrieval Loop

**The problem, from Module 1:** one-shot RAG — embed the question, vector-search once, one LLM
call — works well on single-chunk factual questions but falls apart on questions that need
*multiple, targeted* retrievals across documents. A single top-k=5 vector search over "3M vs
Apple R&D" mostly returns 3M chunks (they happen to score higher) and the model has no way to
go back and specifically look for Apple's numbers.

**What we build in this module:** an agentic retrieval loop with LangGraph that can decide what
to search for, evaluate whether it has enough, go back for more, generate an answer from a
curated knowledge base, and critique its own answer before returning it.

**New components introduced:**
- `retrieval.vector` — semantic search over `Chunk.embedding` (rewritten for Module 2: company
  filtering, tool-shaped return values)
- `retrieval.keyword` — fulltext (Lucene) search over the `chunk_text` index
- `retrieval.graph_nav` — page-range lookup for a known `doc_id`
- `agent.tools` — the three retrieval functions wrapped as LangChain `@tool`s
- `agent.state` — `AgentState` (the graph's shared state) and the two structured grading
  schemas, `RetrievalGrade` and `AnswerGrade`
- `agent.prompts` — prompt builders for each LLM-backed node
- `agent.nodes` — the five node functions (each callable standalone, for inspection)
- `agent.graph` — `build_agent()`, wiring the nodes into a compiled LangGraph `StateGraph`

## 1. Recap — Where the One-Shot Baseline Struggles

Same question that closed out Module 1. Neo4j already holds both filings from that module's
ingestion run, so we can call straight into `qa.baseline.ask` and watch it fail the same way.

In [1]:
from financial_advisor.qa.baseline import ask

HARD_QUESTION = (
    "Compare 3M's and Apple's approach to research and development investment, "
    "based on their 2018 10-Ks."
)

print(ask(HARD_QUESTION, k=5))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

I don’t know — the context you provided includes 3M’s 2018 R&D disclosures but does not include Apple’s 2018 10‑K, so I can’t make a comparison.

For reference, the 3M information in the context shows that in 2018 3M:
- Spent $1.821 billion on research, development and related expenses ($1.253 billion of that on research and development proper).
- Reported R&D as 5.6% of sales in 2018 (down from 5.9% in 2017 and 2016) and noted a $49 million decrease largely due to a divestiture.
- Charges R&D and related expenses to operations when incurred, invests in disruptive-innovation programs, and emphasizes patents as a competitive advantage.


## 2. Three Retrieval Tools

The agent gets three tools instead of a single fixed vector search, each suited to a different
retrieval situation. They're plain LangChain `@tool`-decorated functions in `agent.tools`, thin
wrappers around `retrieval.vector` / `retrieval.keyword` / `retrieval.graph_nav`:

| Tool | Backing index | Best for |
|---|---|---|
| `semantic_search` | `chunk_embedding` (vector) | broad, conceptual, or comparative questions; unsure of exact wording |
| `fulltext_search` | `chunk_text` (Lucene fulltext) | exact terminology, line items, section titles |
| `get_document_pages` | none — direct `Chunk.doc_id` match | pulling full context once a `doc_id` is already known |

All three accept an optional `company_id` filter (`"3M"` / `"APPLE"`) — critical for a
comparison question, since it lets the agent search each company's filing independently instead
of relying on whichever chunks happen to score highest overall. Every returned chunk carries its
`doc_id`, so a `semantic_search` or `fulltext_search` hit is what feeds a later
`get_document_pages` call.

Let's call each one directly, exactly as the agent will.

In [2]:
from financial_advisor.agent.tools import fulltext_search, get_document_pages, semantic_search

# Tools are LangChain @tool objects — .invoke(dict) runs them exactly as the agent will call them
hits = semantic_search.invoke(
    {"query": "3M research and development expenses 2018", "k": 3, "company_id": "3M"}
)
print(f"semantic_search -> {len(hits)} chunk(s)\n")
for h in hits:
    print(f"  [{h['score']:.3f}] {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

semantic_search -> 3 chunk(s)

  [0.882] 3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/15  pages=[7]
    Research and Patents
Research and product development constitutes an important part of 3M's activities and has been a major driver of 3M's sales and profit grow

  [0.875] 3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/55  pages=[28]
    Research, Development and Related Expenses:
R&D in dollars decreased $49 million for full year 2018 when compared to the same period last year. The decrease pri

  [0.856] 3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/133  pages=[65]
    NOTE 1.  Significant Accounting Policies
Research, development and related expenses: These costs are charged to operations in the period incurred and are shown 



In [3]:
text_hits = fulltext_search.invoke(
    {"query": '"research and development" AND Apple~', "k": 3, "company_id": "APPLE"}
)
print(f"fulltext_search -> {len(text_hits)} chunk(s)\n")
for h in text_hits:
    print(f"  [{h['score']:.3f}] {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

fulltext_search -> 2 chunk(s)

  [4.933] APPLE/APPLE_2018_10K.pdf  chunk_id=APPLE/APPLE_2018_10K.pdf/175  pages=[65, 66, 67]
    Note 10 - Segment Information and Geographic Data
The  Company  reports  segment  information  based  on  the  'management'  approach.  The  management  approac

  [4.919] APPLE/APPLE_2018_10K.pdf  chunk_id=APPLE/APPLE_2018_10K.pdf/4  pages=[4]
    Business Strategy
The Company is committed to bringing the best  user  experience  to  its  customers  through  its  innovative  hardware,  software  and  servi



In [4]:
# Given a doc_id already surfaced by one of the tools above, pull specific pages directly
doc_id = hits[0]["doc_id"]
target_pages = hits[0]["pages"]
page_hits = get_document_pages.invoke({"doc_id": doc_id, "pages": target_pages, "limit": 5})

print(f"get_document_pages({doc_id!r}, pages={target_pages}) -> {len(page_hits)} chunk(s)\n")
for h in page_hits:
    print(f"  {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

get_document_pages('3M/3M_2018_10K.pdf', pages=[7]) -> 3 chunk(s)

  3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/15  pages=[7]
    Research and Patents
Research and product development constitutes an important part of 3M's activities and has been a major driver of 3M's sales and profit grow

  3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/16  pages=[7]
    Raw Materials
In 2018, the Company experienced raw material price inflation across most material markets worldwide. In response, the Company continued to deploy

  3M/3M_2018_10K.pdf  chunk_id=3M/3M_2018_10K.pdf/17  pages=[7, 8]
    Environmental Law Compliance
3M's manufacturing operations are affected by national, state and local environmental laws around the world. 3M has made, and plans



## 3. Shared State

LangGraph nodes read and write a single shared `AgentState` (a `TypedDict`, defined in
`agent.state`). Each node returns only the keys it changes; LangGraph merges them into the
running state.

Two fields are worth calling out because they carry the whole design:

- **`growing_knowledge`** (str) — a self-contained, cumulative write-up of every fact needed to
  answer the question, rebuilt each retrieval round by the grading node. This — *not* the raw
  retrieved chunks — is all the answer generator ever sees. Retrieved chunks can be noisy,
  redundant, or only partially relevant; forcing an LLM to distill them into
  `growing_knowledge` first keeps the final answering step focused and cheap.
- **`retrieved_chunks`** (list[dict]) — every unique chunk pulled back so far, deduplicated by
  `id` and accumulated *across* retrieval rounds, so a second round doesn't lose what the first
  one found.

Two Pydantic models drive the two evaluation nodes via `with_structured_output`:

- **`RetrievalGrade`** — `sufficient: bool`, the rewritten `growing_knowledge: str`, and
  `feedback: str` for the next retrieval round.
- **`AnswerGrade`** — `accepted: bool`, `next_action: Literal["retry_answer",
  "retry_retrieval", "end"]`, and `feedback: str`.

`next_action` is the crux of the answer-evaluation step: the grading LLM has to look at
`growing_knowledge` and decide *why* the answer is wrong — data missing (go back to retrieval)
versus data present but misused (go back to answer generation).

In [5]:
from financial_advisor.agent.state import initial_state

state = initial_state(HARD_QUESTION)
for k, v in state.items():
    print(f"  {k}: {v!r}")

  question: "Compare 3M's and Apple's approach to research and development investment, based on their 2018 10-Ks."
  tool_calls: []
  tool_call_log: []
  retrieved_chunks: []
  retrieval_iterations: 0
  growing_knowledge: ''
  retrieval_feedback: ''
  retrieval_sufficient: False
  answer: ''
  answer_attempts: 0
  answer_feedback: ''
  answer_next_action: ''


## 4. The Graph

Five nodes, two LLM-graded decision points. The happy path runs top to bottom once; each side
loop is one place the agent can decide it isn't done yet.

```
                              ┌────────────┐
                              │   START    │
                              └─────┬──────┘
                                    │
                                    ▼
                     ┌──────────────────────────────┐
              ┌─────▶│  1. retriever_strategy_agent   │
              │      │     picks tool(s) to call,     │
              │      │     given feedback so far       │
              │      └───────────────┬────────────────┘
              │                      │ tool_calls
              │                      ▼
              │      ┌──────────────────────────────┐
              │      │  2. call_tools                 │
              │      │     runs semantic_search /      │
              │      │     fulltext_search /           │
              │      │     get_document_pages          │
              │      └───────────────┬────────────────┘
              │                      │ retrieved_chunks (accumulated)
              │                      ▼
              │      ┌──────────────────────────────┐
              │      │  3. evaluate_retrieval          │
              └──────┤     rewrites growing_knowledge, │
               retry │     decides: sufficient?         │
                      └───────────────┬────────────────┘
                                      │ continue (sufficient, or iteration cap)
                                      ▼
                     ┌──────────────────────────────┐
              ┌─────▶│  4. generate_answer            │
              │      │     sees ONLY                   │
              │      │     growing_knowledge —         │
              │      │     no raw chunks               │
              │      └───────────────┬────────────────┘
       retry_answer                  │ answer
   (info was there,                  ▼
    answer missed it)  ┌──────────────────────────────┐
              └────────┤  5. evaluate_answer             │
                        │     correct & complete?         │
                        └──────┬───────────────┬─────────┘
                               │               │
                          end  │               │ retry_retrieval
                               ▼               │ (info genuinely missing)
                            ┌─────┐             │
                            │ END │             └──────────▶ back to node 1
                            └─────┘
```

Three separate loops, each with a different trigger:
1. **`evaluate_retrieval` → `retriever_strategy_agent`** ("retry") — this round's chunks (plus
   everything before them) aren't enough yet; go search again.
2. **`evaluate_answer` → `generate_answer`** ("retry_answer") — the knowledge is there, the
   answer just didn't use it correctly.
3. **`evaluate_answer` → `retriever_strategy_agent`** ("retry_retrieval") — writing the answer
   exposed a gap that grading the retrieval missed; go search again.

Both retrieval and answering have hard iteration caps (`MAX_RETRIEVAL_ITERATIONS`,
`MAX_ANSWER_ATTEMPTS` in `agent.nodes`) so a stubborn question degrades to "best effort" instead
of looping forever.

In [6]:
from financial_advisor.agent.graph import build_agent

agent = build_agent()

# Cross-check the hand-drawn diagram above against LangGraph's own view of the wiring
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retriever_strategy(retriever_strategy)
	call_tools(call_tools)
	evaluate_retrieval(evaluate_retrieval)
	generate_answer(generate_answer)
	evaluate_answer(evaluate_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> retriever_strategy;
	call_tools --> evaluate_retrieval;
	evaluate_answer -. &nbsp;end&nbsp; .-> __end__;
	evaluate_answer -. &nbsp;retry_answer&nbsp; .-> generate_answer;
	evaluate_answer -. &nbsp;retry_retrieval&nbsp; .-> retriever_strategy;
	evaluate_retrieval -. &nbsp;continue&nbsp; .-> generate_answer;
	evaluate_retrieval -. &nbsp;retry&nbsp; .-> retriever_strategy;
	generate_answer --> evaluate_answer;
	retriever_strategy --> call_tools;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. Run It on the Question That Broke Module 1

Every node prints its own progress (see `agent.nodes`), so invoking the compiled graph gives us
a live trace of the loop: which tools got called, whether retrieval was graded sufficient, and
whether the answer was accepted.

In [7]:
result = agent.invoke(initial_state(HARD_QUESTION), {"recursion_limit": 50})

print(f"\nretrieval_iterations: {result['retrieval_iterations']}")
print(f"answer_attempts:      {result['answer_attempts']}")
print(f"chunks retrieved:     {len(result['retrieved_chunks'])}")
print(f"\n{'=' * 80}\nANSWER:\n{'=' * 80}\n{result['answer']}")

[strategy] iteration 1: 2 tool call(s) planned
    - semantic_search({'query': 'research and development strategy investment expenditures goals R&D expense 2018 10-K', 'k': 10, 'company_id': '3M'})
    - semantic_search({'query': 'research and development strategy investment expenditures goals R&D expense 2018 10-K', 'k': 10, 'company_id': 'APPLE'})
[tools] semantic_search({'query': 'research and development strategy investment expenditures goals R&D expense 2018 10-K', 'k': 10, 'company_id': '3M'}) -> 10 chunk(s)
[tools] semantic_search({'query': 'research and development strategy investment expenditures goals R&D expense 2018 10-K', 'k': 10, 'company_id': 'APPLE'}) -> 10 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end

retrieval_iterations: 1
answer_attempts:      1
chunks retrieved:     20

ANSWER:
Brief summary: Both companies treat R&D as strategically important but take different approaches in scale, composition, drivers and recent trends.

- Scale (absolute dollars)
  - Apple: R&D expense of $14,236 million in 2018. (doc_id=APPLE/APPLE_2018_10K.pdf)
  - 3M: core "research and development" $1,253 million in 2018; total "research, development and related expenses" $1,821 million in 2018. (doc_id=3M/3M_2018_10K.pdf)
  - Implication: Apple’s absolute R&D outlay is roughly an order of magnitude larger than 3M’s. (doc_id=APPLE/APPLE_2018_10K.pdf; doc_id=3M/3M_2018_10K.pdf)

- Intensity (R&D as a percent of sales)
  - Apple: R&D ≈ 5% of net sales in 2018. (doc_id=APPLE/APPLE_2018_10K.pdf)
  - 3M: R&D (research, development and related expenses) ≈ 5.6% of sales in 2018. (doc_id=3M/3M_2018_10K.pdf)
  - Implication: both invest a similar share of revenue in R&

Compare that against the Module 1 baseline's answer at the top of this notebook: same question,
same underlying data, but here the strategy agent can issue a `semantic_search` scoped to each
`company_id` independently — so both companies' numbers make it into `growing_knowledge` — and
`evaluate_answer` would have looped back to retrieval had one side been missing.

`growing_knowledge` is what actually reached `generate_answer` — the answer generator never saw
`retrieved_chunks` directly:

In [8]:
print(result["growing_knowledge"])

3M (2018 10‑K, doc_id=3M/3M_2018_10K.pdf):
- Total "research, development and related expenses" = $1.821 billion in 2018; this was $1.870 billion in 2017 and $1.764 billion in 2016 (doc_id=3M/3M_2018_10K.pdf, chunk_id=15/page7). 
- Within that, "research and development" (covering basic scientific research and application of scientific advances to develop new and improved products) = $1.253 billion in 2018; $1.352 billion in 2017; $1.248 billion in 2016 (doc_id=3M/3M_2018_10K.pdf, chunk_id=15/page7).
- R&D (research, development and related expenses) measured as percent of sales = 5.6% in 2018, versus 5.9% in 2017 and 2016; dollar R&D decreased $49 million in 2018 vs 2017, primarily because R&D related to the Communication Markets Division was no longer incurred following its divestiture (doc_id=3M/3M_2018_10K.pdf, chunk_id=55/page28 and chunk_id=52/page27).
- 3M describes R&D as "an important part" of its activities and "a major driver" of sales and profit growth; it emphasizes basic 

## 6. A Few More Questions — Easy vs. Harder

The single-chunk factual questions Module 1 already handled well should still resolve in one
retrieval round here — the agentic loop shouldn't cost extra iterations when it doesn't need
them. The harder, cross-document question is where the extra machinery pays for itself.

In [9]:
EXAMPLE_QUESTIONS = [
    "What is 3M's principal executive office address?",  # single-chunk factual — easy
    "What was Apple's total revenue in fiscal year 2018?",  # single-chunk factual — easy
    "Compare 3M's and Apple's gross margin and operating margin in fiscal 2018, "
    "based on their 10-Ks.",  # cross-document synthesis — hard
]

for q in EXAMPLE_QUESTIONS:
    print(f"\n{'=' * 80}\nQ: {q}\n{'=' * 80}")
    r = agent.invoke(initial_state(q), {"recursion_limit": 50})
    print(
        f"\n[{r['retrieval_iterations']} retrieval round(s), "
        f"{r['answer_attempts']} answer attempt(s), "
        f"{len(r['retrieved_chunks'])} chunk(s) retrieved]"
    )
    print(f"\nA: {r['answer']}")


Q: What is 3M's principal executive office address?


[strategy] iteration 1: 1 tool call(s) planned
    - fulltext_search({'query': 'company:3M AND "principal executive offices" OR "principal executive office"', 'k': 10, 'company_id': '3M'})
[tools] fulltext_search({'query': 'company:3M AND "principal executive offices" OR "principal executive office"', 'k': 10, 'company_id': '3M'}) -> 0 chunk(s)


[grade-retrieval] sufficient=False
    feedback: Missing: any authoritative source that states 3M’s principal executive office address (examples: 3M’s most recent Form 10-K or other SEC filing, the official 3M corporate website/contact or investor relations page, or a recent press release that lists the corporate headquarters). Next steps: use a web/filing search tool with queries such as "3M principal executive office address", "3M principal executive office address 10-K", or retrieve 3M’s latest Form 10-K (SEC EDGAR) and extract the principal executive office address from the business or corporate governance section. Preferred documents to retrieve: 3M Company Form 10-K (latest year) on SEC EDGAR, or 3M corporate website 'Contact Us' (official headquarters address).


[strategy] iteration 2: 1 tool call(s) planned
    - semantic_search({'query': 'principal executive office address 3M principal executive offices 3M Center St. Paul Minnesota 55144', 'k': 10, 'company_id': '3M'})
[tools] semantic_search({'query': 'principal executive office address 3M principal executive offices 3M Center St. Paul Minnesota 55144', 'k': 10, 'company_id': '3M'}) -> 10 chunk(s)


[grade-retrieval] sufficient=True


...

[grade-answer] accepted=True next_action=end

[2 retrieval round(s), 1 answer attempt(s), 10 chunk(s) retrieved]

A: Apple’s total revenue (reported as “Net sales”) for fiscal year 2018 was $265,595 million (Net sales, Years ended September 29, 2018 = $265,595) (doc_id=APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/117; doc_id=APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/175). The Revenue Recognition note confirms that “Net sales” corresponds to the Company’s total revenue (doc_id=APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/124).

Q: Compare 3M's and Apple's gross margin and operating margin in fiscal 2018, based on their 10-Ks.


[strategy] iteration 1: 2 tool call(s) planned
    - semantic_search({'query': 'gross margin operating margin fiscal 2018 gross margin 2018 operating margin 2018 10-K', 'k': 5, 'company_id': '3M'})
    - semantic_search({'query': 'gross margin operating margin fiscal 2018 gross margin 2018 operating margin 2018 10-K', 'k': 5, 'company_id': 'APPLE'})
[tools] semantic_search({'query': 'gross margin operating margin fiscal 2018 gross margin 2018 operating margin 2018 10-K', 'k': 5, 'company_id': '3M'}) -> 5 chunk(s)
[tools] semantic_search({'query': 'gross margin operating margin fiscal 2018 gross margin 2018 operating margin 2018 10-K', 'k': 5, 'company_id': 'APPLE'}) -> 5 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end

[1 retrieval round(s), 1 answer attempt(s), 10 chunk(s) retrieved]

A: Yes — the 10‑Ks give a clear comparison for fiscal 2018.

Gross margin
- Apple: gross margin $101,839 million, gross margin percentage 38.3% (FY2018) (APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/84).  
- 3M: cost of sales 50.9% of sales → gross margin 100% − 50.9% = 49.1% (FY2018) (3M/3M_2018_10K.pdf, chunk_id=3M/3M_2018_10K.pdf/52).

Result: 3M’s gross margin (≈49.1%) was materially higher than Apple’s (38.3%) in fiscal 2018 (3M higher by ~10.8 percentage points) (3M/3M_2018_10K.pdf, chunk_id=3M/3M_2018_10K.pdf/52; APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/84).

Operating margin
- Apple: net sales $265,595 million and operating income $70,898 million → implied operating margin ≈ 70,898 / 265,595 ≈ 26.7% (FY2018) (APPLE/APPLE_2018_10K.pdf, chunk_id=APPLE/APPLE_2018_10K.pdf/117).  
- 3M: reported operating income margin 22.0% (FY2018); op

## 7. Where This Still Struggles

The loop is only as good as the graph it searches. `Company → Document → Chunk` is text sliced
out of 10-Ks — it has no notion of *people* and how they connect across companies. 3M's 10-K
does list its executive officers, but for their biographical history it explicitly defers to a
document we never ingested: 3M's proxy statement. Watch what happens when we ask for exactly
that.

In [10]:
STRUGGLE_QUESTION = (
    "Which other companies have 3M's current executive officers previously worked at or "
    "served as directors of?"
)

r = agent.invoke(initial_state(STRUGGLE_QUESTION), {"recursion_limit": 50})
print(
    f"\n[{r['retrieval_iterations']} retrieval round(s), "
    f"{r['answer_attempts']} answer attempt(s), "
    f"{len(r['retrieved_chunks'])} chunk(s) retrieved]"
)
print(f"\nA: {r['answer']}")

[strategy] iteration 1: 1 tool call(s) planned
    - semantic_search({'query': "executive officers of the registrant previous employment 3M current executive officers biographies directors served as companies 3M 'Executive Officers' 'Biographical'", 'k': 10, 'company_id': '3M'})
[tools] semantic_search({'query': "executive officers of the registrant previous employment 3M current executive officers biographies directors served as companies 3M 'Executive Officers' 'Biographical'", 'k': 10, 'company_id': '3M'}) -> 10 chunk(s)


[grade-retrieval] sufficient=False
    feedback: Missing information needed to answer the question: specific external companies (outside 3M) that each current 3M executive officer previously worked at, and any other public-company boards on which they have served as directors. To obtain this, retrieve the 3M Proxy Statement (DEF 14A) for the 2019 annual meeting (May 14, 2019) — search for sections titled 'Directors and Executive Officers,' 'Proposal No. 1,' or individual director/executive biographies — and Item 1 (Executive Officers) of 3M’s Form 10-K. If DEF 14A (the proxy statement) is not available, query the 3M investor relations site for executive/board biographies or look up each officer’s biography (e.g., LinkedIn or company bio pages) to identify prior employers and directorships.


[strategy] iteration 2: 1 tool call(s) planned
    - semantic_search({'query': "3M Proxy Statement 2019 directors and executive officers biographies DEF 14A May 14 2019 'Directors and Nominees' 'Executive Officers' biographies", 'k': 10, 'company_id': '3M'})
[tools] semantic_search({'query': "3M Proxy Statement 2019 directors and executive officers biographies DEF 14A May 14 2019 'Directors and Nominees' 'Executive Officers' biographies", 'k': 10, 'company_id': '3M'}) -> 10 chunk(s)


[grade-retrieval] sufficient=False
    feedback: Not sufficient. Missing the external-biography details (past employers and outside directorships) for 3M's listed executive officers and directors. Next steps: retrieve the 3M Proxy Statement (Definitive Proxy, Form DEF 14A) for the 2019 annual meeting (filed with the SEC, typically April 2019) — specifically the 'Proposal No. 1' / 'Directors and Nominees' and 'Executive Officers' biography sections — which are incorporated by reference into the 2018 Form 10-K and are expected to list other companies where these executives have worked or served as directors. If DEF 14A is unavailable, retrieve Item 1 of the 2018 Form 10-K (full Item 1 text) and any separate biographical attachments. Search queries to try next: '3M DEF 14A 2019', '3M proxy statement 2019 PDF', or '3M proxy May 14 2019 DEF 14A'.


[strategy] iteration 3: 1 tool call(s) planned
    - semantic_search({'query': "3M DEF 14A 2019 proxy statement directors and nominees executive officers biographies May 14 2019 'DEF 14A' 'Proxy Statement' 'Directors and Nominees' 'Executive Officers'", 'k': 10, 'company_id': '3M'})
[tools] semantic_search({'query': "3M DEF 14A 2019 proxy statement directors and nominees executive officers biographies May 14 2019 'DEF 14A' 'Proxy Statement' 'Directors and Nominees' 'Executive Officers'", 'k': 10, 'company_id': '3M'}) -> 10 chunk(s)


[grade-retrieval] sufficient=False
    feedback: Missing information needed to answer the question precisely: for each listed 3M executive officer (Inge Thulin; Michael Roman; John Banovetz; James Bauman; Julie Bushman; Joaquin Delgado; Ivan Fong; Nicholas Gangestad; Eric Hammes; Paul Keel; Ashish Khandpur; Jon Lindekugel; Kristen Ludgate; Mojdeh Poul; Michael Vale) the 10-K does not provide their prior external employers or outside directorships. To obtain these, retrieve the 3M 2019 definitive proxy statement (DEF 14A) — the proxy statement for the May 14, 2019 annual meeting — which the 10-K incorporates by reference and which contains directors’ and executive officers’ biographies and outside affiliations. Suggested next actions (pick one or more):

- Fetch 3M’s definitive proxy statement (DEF 14A) for 2019 (proxy for annual meeting May 14, 2019) from SEC EDGAR or 3M’s investor relations site.
- Alternatively, retrieve the full Item 1 executive officer biographies from the 2018 For

[strategy] iteration 4: 1 tool call(s) planned
    - fulltext_search({'query': '3M AND ("DEF 14A" OR "Proxy Statement" OR "definitive proxy") AND 2019', 'k': 10, 'company_id': '3M'})
[tools] fulltext_search({'query': '3M AND ("DEF 14A" OR "Proxy Statement" OR "definitive proxy") AND 2019', 'k': 10, 'company_id': '3M'}) -> 1 chunk(s)


[grade-retrieval] sufficient=False
    feedback: Missing information needed to answer question precisely: the executive officers’ prior employers and the companies on whose boards they have served (outside directorships). These details are not present in the retrieved 2018 Form 10-K chunks. Exact next steps to obtain the missing facts:

1) Retrieve 3M’s definitive proxy statement (Form DEF 14A) for the 2019 annual meeting (May 14, 2019). That proxy (the 2019 DEF 14A) contains director/nominee biographies and usually lists outside directorships and prior affiliations — search SEC EDGAR for "3M Company DEF 14A 2019" or fetch the filing for the 2019 annual meeting. (Target: 2019 DEF 14A / 3M Proxy Statement.)

2) Retrieve the full Item 1 (Executive Officers) section of the 2018 Form 10-K or the stand-alone executive officer biographies on 3M’s investor or corporate website (investors.3M.com or www.3M.com) for any additional employment history not in the proxy. (Target: Item 1 full text or

[answer] attempt #1


[grade-answer] accepted=True next_action=end

[4 retrieval round(s), 1 answer attempt(s), 17 chunk(s) retrieved]

A: I cannot determine which outside companies these 3M executive officers previously worked at or have served as directors of from the materials you provided.

What the provided 2018 Form 10-K does contain
- The 10-K lists the executive officers (as of Feb 7, 2019): Inge G. Thulin; Michael F. Roman; John P. Banovetz; James L. Bauman; Julie L. Bushman; Joaquin Delgado; Ivan K. Fong; Nicholas C. Gangestad; Eric D. Hammes; Paul A. Keel; Ashish K. Khandpur; Jon T. Lindekugel; Kristen M. Ludgate; Mojdeh Poul; and Michael G. Vale. (doc_id=3M/3M_2018_10K.pdf, chunk_id=18)

Why I cannot answer from the provided material
- The retrieved 2018 Form 10-K excerpts do not list other (external) companies where those named executive officers previously worked or where they have served as directors; the 10-K indicates that such director/nominee and executive-officer detail is contained in t

However many times the loop retries, `semantic_search` and `fulltext_search` can only surface
what the corpus actually contains — chunks of the 10-K itself, which name the executive officers
but explicitly point elsewhere ("incorporated by reference" to the proxy statement) for their
career history at other companies. `evaluate_retrieval` will keep asking for another round, hit
`MAX_RETRIEVAL_ITERATIONS`, and `generate_answer` is left honestly saying the knowledge isn't
there — which is the correct behavior for a retrieval system, just not a useful answer. This
isn't a bug in the loop; it's a graph coverage gap. `get_document_pages` can't page through a
document that was never ingested.

## 8. What's Next

Module 3 enriches the graph with `Person`, `Event`, and `Article` nodes (`ROLE_AT`,
`MENTIONED_IN`) — pulling in exactly the executive-history and news data the previous section's
question needed. The retrieval loop built here stays as-is; it just gets more of the graph, and
more tools, to work with.